CRN parameters

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import psutil
from sklearn import datasets
from sklearn.model_selection import train_test_split
from itertools import combinations

def subsets(I, n):
    return np.array(list(combinations(range(I), n)))


#CRN parameters

I = 64#number of input species
K = 10 #number of classes
M_sel = 40 #number of images to select fluxes

Tlearn = 1
Tren = 10

n = 1 #network depth
#n = 2

#to use thresholds instead of selecting a specific number of species
#theta = 0.95
#rho = 1 

#initial concentrations and rate constants
b1 = 1
b2 = 1
a0 = 1
s0 = 3
a2 = 1

#creation of set J_n
J_n = subsets(I,n) #(nb of subsets, n)
nb_J_n = J_n.shape[0] #nb of indices j before selection

#learning rate
eta = 0.0005 #for n = 1
#eta = 0.0001 #for n = 2


This is the simplified version, where weights are renormalized only once and output species are produced only during test phase

CRN implementation (selection phase, learning phase, testing phase). Since the exact solutions of the rate equations can be computed in this context, we compute the exact values.

In [ ]:
#the only randomness is on the splitting of the dataset and the noise

exp = 1 #numerotation of experiments to save data
#exp = 2 #for n = 2

Nsimu = 100 #number of simulations

nb_max_sel = nb_J_n

accuracy = np.zeros((nb_max_sel,Nsimu))

for p in range(Nsimu): #loop on the simulations
    for nb_sel in range(1,nb_max_sel): #results for each number of selectec sets

        #retrieve dataset
        digits = datasets.load_digits()
        target=digits.target #(1797,)

        #randomly split between train and test sets
        X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.2)
        M_train = np.shape(X_train)[0] #1437
        M_test = np.shape(X_test)[0] #360
        M_out = M_train - M_sel #number of images to train output weights


        # Reshape images 
        flattened_images_train = X_train.reshape((M_train, -1))
        flattened_images_test = X_test.reshape((M_test, -1))

        images_train = flattened_images_train /16
        images_test = flattened_images_test /16

        #add noise in the input signals
        var = 0.00001
        noise_train = np.random.normal(loc=0.0, scale=np.sqrt(var), size=(M_train, 64))
        noise_test = np.random.normal(loc=0.0, scale=np.sqrt(var), size=(M_test, 64))

        images_train =np.maximum(images_train + noise_train,0) #(1797,8,8)
        images_test =np.maximum(images_test + noise_test,0)

        target_train = y_train #sequence of digits from 0 to 9
        target_test = y_test
        

        #selection phase
        flux = np.zeros((nb_J_n, M_sel))
        images_sel = images_train[:M_sel, :]

        #  compute all fluxes
        for r in range(nb_J_n):
            flux[r] = np.prod(images_sel[:, J_n[r,:]], axis=1)

        #selection of fluxes

        #with threshold theta
        #ind_sel = np.where(np.max(flux, axis=1) > theta)[0]

        #with a predefined number of slected fluxes
        ind_sel = np.argsort(np.max(flux, axis=1))[-nb_sel:]

        J = J_n[ind_sel,:]
        flux_sel = flux[ind_sel, :]
        nJ = J.shape[0]

        #selection network, explicit weight formula at the end
        WJ = np.sum(flux_sel* Tlearn, axis = 1)

        #renormalization network, value at the end
        WJ = (b1/b2) + (WJ - (b1/b2)) * np.exp(- b2 * a0 * Tren)


        #learning phase

        #compute fluxes
        flux_learn = np.zeros((nJ, M_out))
        images_learn = images_train[M_sel:, :]
        for l in range(nJ):
            flux_learn[l] = np.prod(images_learn[:, J_n[ind_sel[l],:]], axis=1)


        images_out = images_train[M_sel:,:] #(M_out, 64)
        #correct class
        k_star = y_train[M_sel:] #(M_out)

        #compute gains
        g =  np.zeros((nJ,K, M_out)) #contains gains
        for m in range(M_out):
            #wrong classes
            for k in range(K):
                if k != k_star[m]:
                    g[:,k,m] = (10/9) * Tlearn *  WJ * ( - flux_learn[:,m] + s0) 

            #correct class
            g[:,k_star[m],m] = WJ * Tlearn * (flux_learn[:,m] + s0) * 10

        G = np.sum(g, axis = 2)

        #EWA network, value at the end of the learning phase
        H = np.exp(eta * G) #(nJ, K)

        #Renormalization network, once learning is finished
        WK_bar = H / np.sum(H, axis = 0)
        WK = (b1/b2)  * WK_bar + ( H - (b1/b2) *  WK_bar) * np.exp(-b2 * Tren * np.sum(H, axis = 0)) #(nJ,K)


        #Test phase
        y_hat = np.zeros(M_test) #network classifications
        flux_test = np.zeros((nJ, M_test))


        for l in range(nJ):
            flux_test[l] = np.prod(images_test[:, J_n[ind_sel[l],:]], axis=1)

        X = np.zeros(K) #initial output concentrations
        for m in range(M_test):
            #Forward pass equation
            prod = WJ * flux_test[:,m]
            X  +=  np.einsum('j, jk -> k', prod, WK)
            #print(X)

            #network decision
            y_hat[m] = np.argmax(X)

            #network decay
            X = X * np.exp(-a2*Tren)
            #print(X)

        accuracy[nb_sel-1,p] = 100 * np.sum(y_hat == y_test) / M_test
    print(p)

np.save(f"/{exp}_n", n)
np.save(f"/{exp}_eta", eta)
np.save(f"/{exp}_accuracy", accuracy)


To generate Figure 3

In [ ]:
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

exp1 = 1
exp2 = 2

accuracy1 = np.load(f"/{exp1}_accuracy.npy")
accuracy2 = np.load(f"/{exp2}_accuracy.npy")

Nsimu = accuracy1.shape[1]
nb_max_sel1 = accuracy1.shape[0]
accuracy1 = accuracy1[:nb_max_sel1-1]
moy_acc1 = np.mean(accuracy1, axis = 1)

level = 0.1 #level confidence interval
sorted_acc1 = np.sort(accuracy1, axis = 1)
inf_acc1 = sorted_acc1[:,int(Nsimu * level)]
sup_acc1 = sorted_acc1[:,int(Nsimu * (1-level))]


fig, axs = plt.subplots(
    1, 2,
    figsize=(8, 3),
    sharey = True,
    gridspec_kw={'width_ratios': [1, 2]}  # gauche 2x plus large que droite
)

Z1 =np.arange(1,nb_max_sel1)

axs[0].plot(Z1, moy_acc1, label = r"Network depth $n = 1$")
axs[0].fill_between(Z1, inf_acc1, sup_acc1, color = 'blue', alpha = .2)


nb_max_sel2 = accuracy2.shape[0]
accuracy2 = accuracy2[:nb_max_sel2-1]
moy_acc2 = np.mean(accuracy2, axis = 1)

sorted_acc2 = np.sort(accuracy2, axis = 1)
inf_acc2 = sorted_acc2[:,int(Nsimu * level)]
sup_acc2 = sorted_acc2[:,int(Nsimu * (1-level))]

Z2 =np.arange(1,nb_max_sel2)


axs[1].plot(Z2, moy_acc2, label = r"Network depth $n = 2$")
axs[1].fill_between(Z2, inf_acc2, sup_acc2, color = 'blue', alpha = .2)
axs[0].set_xlabel(r'Number of selected sets $|\bar{J}_1|$')
axs[1].set_xlabel(r'Number of selected sets $|\bar{J}_2|$')
axs[0].set_ylabel(r'Test accuracy ($\%$)')
#axs.set_title('Test accuracy depending on the number of selected sets')
axs[0].legend()
axs[1].legend()
plt.tight_layout()
#plt.savefig('accuracy.pdf')
plt.show()

To plot examples of handwritten digits

In [ ]:
plt.rc('text', usetex=True)
plt.rc('font', family='serif')
digits = datasets.load_digits()
(n_samples, n_features) = digits.data.shape
print(f"Dataset dimensions: {(n_samples, n_features)}")

# Display a few digits
fig, axs = plt.subplots(2, 3, figsize=(4, 4))
idx = [0, 1, 2, 3, 4, 5]
for i, ax in enumerate(axs.ravel()):
    ax.imshow(digits.images[idx[i]], cmap='gray')
    ax.set_title(f"Label {digits.target[idx[i]]}")
    ax.axis('off')
plt.tight_layout()
plt.show()